# NeuroScan AI — Sprint 2
## Model Fine-Tuning ve Grad-CAM (Açıklanabilir Yapay Zeka)

**Takım:** NeuroVision (Takım 71) · **Sorumlu:** Pelşin Gündüz (ML/DL Developer)  
**Sprint:** 2 (6 – 19 Temmuz 2026)

### Bu notebook'un amacı
Sprint 2 kapsamında eğitilen **v2 modelinin** performansını raporlamak ve modelin
MR görüntüsünde **hangi bölgeye odaklandığını** Grad-CAM ile görselleştirmek.

### Mimari notu
Eğitim kodu bu notebook'un içinde **değil**, yeniden kullanılabilir modüllerdedir:

| Dosya | Görevi |
|---|---|
| `src/labels.py` | Sınıf sırasının tek kaynağı (single source of truth) |
| `src/train_finetune.py` | 2 fazlı fine-tuning + metrik üretimi |
| `src/gradcam.py` | Grad-CAM heatmap ve overlay fonksiyonları |

Bu notebook o modülleri **çağırır**. Aynı `gradcam.py` Taha'nın Streamlit
uygulamasında da import edilerek kullanılır — böylece notebook ile uygulama
arasında davranış farkı oluşmaz.

> ⚠️ **Medikal uyarı:** Bu proje klinik tanı koymaz. Eğitim ve karar destek
> prototipidir. Grad-CAM çıktısı modelin dikkatini gösterir; klinik lokalizasyon
> veya tanı değildir.

## 1. Kurulum ve model yükleme

In [ ]:
import sys, os, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Repo kökünü path'e ekle (notebook notebooks/ içinde çalışıyor)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.labels import CLASS_NAMES, DISPLAY_NAMES, display
from src.gradcam import make_gradcam_heatmap, overlay_heatmap

print("Repo kökü:", ROOT)
print("Sınıf sırası:", CLASS_NAMES)

In [ ]:
MODEL_PATH = ROOT / "models" / "neuroscan_mobilenetv2_v2.keras"
model = tf.keras.models.load_model(MODEL_PATH)

print("Model yüklendi:", MODEL_PATH.name)
print("Girdi shape :", model.input_shape)
print("Çıktı shape :", model.output_shape)

## 2. Fine-tuning stratejisi

`src/train_finetune.py` iki fazlı bir eğitim uygular:

| Faz | Backbone | Learning Rate | Epoch | Amaç |
|---|---|---|---|---|
| **1** | Donuk (frozen) | 1e-3 | 8 | Sınıflandırıcı head'i ImageNet özellikleri üzerinde eğitmek |
| **2** | Son 40 katman açık | **1e-5** | 12 | Backbone'u MR görüntülerine uyarlamak (fine-tuning) |

**Neden iki faz?** Backbone'u en baştan açıp yüksek LR ile eğitmek, rastgele
başlatılmış head'den gelen büyük gradyanların ImageNet ağırlıklarını bozmasına
yol açar. Önce head oturur, sonra çok düşük LR ile backbone ince ayarlanır.

**Ek olarak uygulananlar:**
- `class_weight` — sınıf dengesizliğini telafi eder (glioma 1932 vs meningioma 1230)
- Augmentation — flip, rotation, zoom, contrast (yalnızca eğitim setinde)
- `EarlyStopping` + `ModelCheckpoint` — en iyi val_accuracy'li model saklanır

## 3. Sonuçlar: Baseline vs v2

In [ ]:
# Eğitim eğrileri ve confusion matrix (train_finetune.py tarafından üretildi)
for name in ["training_curves_v2.png", "confusion_matrix_v2.png"]:
    path = ROOT / "assets" / "model_results" / name
    img = plt.imread(path)
    plt.figure(figsize=(11, 5))
    plt.imshow(img); plt.axis("off"); plt.title(name)
    plt.show()

In [ ]:
report_path = ROOT / "assets" / "model_results" / "classification_report_v2.txt"
print(report_path.read_text(encoding="utf-8"))

### 3.1 Karşılaştırma ve yorum

| Metrik | Sprint 1 (baseline) | Sprint 2 (v2) |
|---|---|---|
| Test Accuracy | 0.7911 | **0.8470** |
| Validation Accuracy | — | 0.9281 |

**Fine-tuning kazancı:** Test accuracy %79.1 → %84.7 (**+5.6 puan**).

#### Dikkat edilmesi gereken üç nokta

**1. Validation (%92.8) ile Test (%84.7) arasındaki fark.**  
Validation seti `train` klasöründen ayrılmıştır; aynı hastanın komşu MR kesitleri
hem train hem validation'a düşmüş olabilir. Test seti ise tamamen ayrı bir
klasördür. Bu nedenle **gerçekçi performans göstergesi test setidir (%84.7)**;
validation skoru iyimser taraflıdır.

**2. Glioma recall'ü düşük (0.73).**  
Precision 0.96 — model "glioma" dediğinde neredeyse her zaman haklı. Ancak
gliomaların %27'sini kaçırıyor. Model glioma sınıfında **temkinli** davranıyor.

**3. En kritik hata: `no` sınıfının precision'ı 0.83.**  
Model, gerçekte tümörlü olan bir kısım görüntüye "tümör yok" diyor
(false negative). Medikal bağlamda **en riskli hata tipi budur.** Aşağıdaki
hücrede bu hatanın büyüklüğü sayısal olarak hesaplanmıştır.

Bu bulgu, ürünün klinik tanı yerine geçemeyeceğine dair uyarının neden zorunlu
olduğunu somut olarak göstermektedir.

In [ ]:
# En kritik hata: gerçekte tümörlü olup "tümör yok" tahmin edilen vakalar
from sklearn.metrics import confusion_matrix

DATA_DIR = ROOT / "notebooks" / "Brain-Tumour-1"
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=(224, 224), batch_size=32,
    label_mode="categorical", shuffle=False,
)
assert list(test_ds.class_names) == CLASS_NAMES, "Sınıf sırası uyuşmuyor!"

y_true, y_pred = [], []
for images, lbls in test_ds:
    p = model.predict(images, verbose=0)
    y_true.extend(np.argmax(lbls.numpy(), axis=1))
    y_pred.extend(np.argmax(p, axis=1))
y_true, y_pred = np.array(y_true), np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
no_idx = CLASS_NAMES.index("no")

tumor_mask = y_true != no_idx                      # gerçekte tümörlü olanlar
missed = int(np.sum((y_true != no_idx) & (y_pred == no_idx)))
total_tumor = int(tumor_mask.sum())

print(f"Test setindeki tümörlü görüntü sayısı : {total_tumor}")
print(f"'Tümör yok' olarak yanlış sınıflananlar: {missed}")
print(f"KAÇIRILAN TÜMÖR ORANI (false negative) : {missed/total_tumor:.2%}")
print()
print("Bu oran, ürünün klinik tanı aracı olarak kullanılamayacağının")
print("en somut kanıtıdır ve medikal uyarının gerekçesidir.")

## 4. Grad-CAM — Açıklanabilir Yapay Zeka

**Grad-CAM nedir?** Modelin bir sınıfa karar verirken görüntünün hangi bölgesine
"baktığını" gösteren ısı haritası. Son konvolüsyon katmanının aktivasyonlarını,
tahmin edilen sınıfın skoruna göre alınan gradyanlarla ağırlıklandırarak üretilir.

**Neden değerli?** Medikal görüntülemede sadece "glioma" demek yeterli değildir;
modelin **tümör bölgesine mi yoksa alakasız bir artefakta mı** odaklandığı
güvenilirliğin göstergesidir.

### Bu projeye özel iki teknik zorluk

`src/gradcam.py` iki tuzağı çözer:

1. **`data_augmentation` bir `Sequential`'dır ve `Sequential` da `tf.keras.Model`
   alt sınıfıdır.** "Model olan ilk katmanı backbone say" yaklaşımı yanlışlıkla
   augmentation katmanını yakalar. Kod Sequential'ları eler.

2. **Keras 3'te nested alt-modelin `output` tensörü ana modelin grafiğine bağlı
   değildir.** İnternetteki klasik
   `Model(inputs=model.inputs, outputs=[base.output, model.output])` kurulumu
   *"Output is not connected to inputs"* hatası verir. Kod bunun yerine
   backbone'u ayrı çağırıp head katmanlarını elle üstüne uygular.

> **Önemli:** `preprocess_input` modelin **içinde** olduğu için Grad-CAM'e
> **ham 0-255 görüntü** verilir. Dışarıdan preprocess uygulanmaz.

In [ ]:
GRADCAM_DIR = ROOT / "assets" / "model_results" / "gradcam_examples"
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)

def gradcam_for(image_path, save_name=None):
    """Tek bir MR görüntüsü için Grad-CAM üretir ve gösterir."""
    img = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    raw = tf.keras.utils.img_to_array(img)          # 0-255, preprocess YOK
    batch = np.expand_dims(raw, axis=0)

    heatmap, preds = make_gradcam_heatmap(batch, model)
    idx = int(np.argmax(preds))
    label, conf = CLASS_NAMES[idx], float(preds[idx])
    overlay = overlay_heatmap(raw, heatmap)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
    axes[0].imshow(raw.astype("uint8")); axes[0].set_title("Orijinal MR")
    axes[1].imshow(heatmap, cmap="jet");  axes[1].set_title("Grad-CAM (7x7)")
    axes[2].imshow(overlay)
    axes[2].set_title(f"{display(label)} — %{conf*100:.1f}")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()

    # Tüm sınıf olasılıkları
    for c, p in sorted(zip(CLASS_NAMES, preds), key=lambda t: -t[1]):
        print(f"  {display(c):28s} %{p*100:5.2f}")

    if save_name:
        overlay.save(GRADCAM_DIR / save_name)
        print("Kaydedildi:", save_name)
    return label, conf

In [ ]:
# Her sınıftan birer örnek üzerinde Grad-CAM çalıştır
TEST_DIR = DATA_DIR / "test"

for cls in CLASS_NAMES:
    files = sorted(
        f for f in (TEST_DIR / cls).iterdir()
        if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )
    sample = files[0]

    print("=" * 70)
    print(f"GERÇEK SINIF: {display(cls)}   ({sample.name})")
    print("=" * 70)
    gradcam_for(sample, save_name=f"gradcam_{cls}.png")
    print()

### 4.1 Grad-CAM yorumu

*(Bu bölümü yukarıdaki çıktılara bakarak kendi gözlemlerinle doldur.)*

Değerlendirirken şunlara bak:
- **Isı haritası tümör bölgesiyle örtüşüyor mu**, yoksa kafatası kenarı / arka
  plan gibi alakasız bir bölgeye mi odaklanmış?
- `no` (tümör yok) sınıfında model **nereye** bakıyor? Beklenen: dağınık veya
  merkezi, belirgin bir odak yok.
- Modelin **yanlış tahmin ettiği** bir örnekte ısı haritası nereye kaymış?
  (Hata analizi için güçlü bir kanıt olur.)

> **Uyarı:** Grad-CAM modelin dikkatini gösterir. **Tümörün klinik konumunu
> belirlemez ve segmentasyon değildir.** Kullanıcı arayüzünde bu ayrım net
> biçimde belirtilmelidir.

## 5. Sprint 2 çıktıları — kontrol listesi

Bu notebook ve `src/train_finetune.py` şu dosyaları üretti:

| Dosya | Açıklama |
|---|---|
| `models/neuroscan_mobilenetv2_v2.keras` | Fine-tune edilmiş v2 model |
| `models/labels.json` | Sınıf sırası (app için tek kaynak) |
| `assets/model_results/training_curves_v2.png` | Accuracy / loss eğrileri |
| `assets/model_results/confusion_matrix_v2.png` | Confusion matrix |
| `assets/model_results/classification_report_v2.txt` | Precision / recall / F1 |
| `assets/model_results/gradcam_examples/` | 4 sınıf için Grad-CAM overlay |
| `docs/model_report_v2.md` | Model raporu v2 |

### Taha'ya devir notu (app entegrasyonu)

1. Sınıf listesini **elle yazma** — `models/labels.json`'ı `src.labels.load_labels()` ile oku.
2. Modele **ham 0-255 görüntü** ver; `preprocess_input` model içinde uygulanıyor.
3. Grad-CAM için aynı `src/gradcam.py` fonksiyonlarını import et:
   `make_gradcam_heatmap(batch, model)` → `(heatmap, predictions)` döner.
4. Güven skoru %60'ın altındaysa arayüzde "Model bu tahminde emin değil" uyarısı göster.

---

> ⚠️ **Medikal uyarı:** NeuroScan AI klinik tanı koymaz. Eğitim ve demo amaçlı
> bir karar destek prototipidir. Test setinde ölçülen kaçırılan tümör oranı
> (yukarıda hesaplanmıştır), sistemin bir hekimin değerlendirmesinin yerine
> geçemeyeceğini açıkça göstermektedir. Lütfen hekiminize başvurunuz.